In [49]:
import numpy as np
import pandas as pd

In [50]:
def create_df(initial_df):
    temp_df= []
    for id_, group in initial_df.groupby('id'):
        music_sentence= ''

        last_note=-10
        for _, row in group.iterrows():
            if last_note== -10:
                music_sentence+= '0_' + str(row['duration'])
                last_note= row['pitch']
                continue
            
            music_sentence+= ' '+ str(int(row['pitch']-last_note))+ '_'+ str(row['duration'])
            last_note= row['pitch']
        
        temp_df.append({
            'datapointID': id_,
            'music': music_sentence
        })

    return pd.DataFrame(temp_df)

In [51]:
train_df= create_df(pd.read_csv('train_data/train_data.csv'))
train_df= train_df.merge(pd.read_csv('train_data/composers.csv'),
    left_on="datapointID",
    right_on="datapointID")
train_df.drop(columns='subtaskID', inplace= True)

test_df= create_df(pd.read_csv('test_data.csv'))

In [52]:
from sklearn.preprocessing import LabelEncoder

le= LabelEncoder()

X_train= train_df['music']
y= le.fit_transform(train_df['answer'])

X_test= test_df['music']

In [ ]:
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

features= FeatureUnion([
    ('words', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1,4),
        min_df=2,
        max_df=0.9
    ))
])

model= Pipeline([
    ('features', features),
    ('regressor', LogisticRegression(max_iter= 100))
])

In [54]:
model.fit(X_train, y)

predictions= le.inverse_transform(model.predict(X_test))

answer= []

for id_, pred in zip(test_df['datapointID'], predictions):
    answer.append({
        'subtaskID': 1,
        'datapointID': id_,
        'answer': pred
    })

pd.DataFrame(answer).to_csv("submission.csv", index= False)